# ScoutTrainer — GPU pipeline on Colab

Runs the heavy perception stage on Colab's free GPU (minutes instead of hours), then hands you results to browse in your **local** dashboard.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

**Nothing to upload** — cell 1 clones the repo. Push your local changes first (`git add -A && git commit -m ... && git push`), otherwise Colab runs the old code.

**YouTube URLs usually fail on Colab** (datacenter IPs get bot-checked). Uploading the video file in step 3A is the reliable route.

**Reference points are optional** — without them the pipeline runs in camera-relative mode and rates players on on-ball actions.

In [ ]:
# 1) Clone the project (re-run any time to pick up pushed changes)
!rm -rf /content/scout
!git clone https://github.com/kpputhiyattil/scout-agent.git /content/scout
%cd /content/scout
!git log --oneline -1
!nvidia-smi -L

In [ ]:
# 2) Install deps (Colab already has torch+CUDA and ffmpeg). Deno = JS runtime for yt-dlp.
!pip -q install -e ".[perception]" 2>&1 | tail -1
!curl -fsSL https://deno.land/install.sh | DENO_INSTALL=/usr/local sh -s -- -y >/dev/null 2>&1
import torch; print('CUDA available:', torch.cuda.is_available())

## 3) Choose ONE source cell — 3A is recommended

In [ ]:
# 3A) RECOMMENDED — upload a video file from your PC (no YouTube blocking)
from google.colab import files
import pathlib, shutil, re
vdir = pathlib.Path('data/videos'); vdir.mkdir(parents=True, exist_ok=True)
vid = files.upload()                                 # 'Choose Files' button appears below
name = next(iter(vid))
safe = re.sub(r'[^A-Za-z0-9._-]+', '_', name)        # spaces break shell arguments
shutil.move(name, vdir / safe)
SOURCE = ['--file', str(vdir / safe)]
print('using', vdir / safe)

In [ ]:
# 3B) YouTube URL — often blocked on Colab, try 3C if it fails
SOURCE = ['--url', 'https://www.youtube.com/watch?v=E9GLiV_jfro']  # <-- change me

In [ ]:
# 3C) YouTube URL + cookies (clears most bot checks)
#     Export cookies.txt from a logged-in browser with a 'Get cookies.txt' extension.
#     Use a throwaway Google account — cookies are login credentials.
import os
from google.colab import files
ck = files.upload()                                  # pick cookies.txt
os.environ['SCOUT_YTDLP_COOKIES'] = '/content/' + next(iter(ck))
SOURCE = ['--url', 'https://www.youtube.com/watch?v=E9GLiV_jfro']  # <-- change me

In [ ]:
# 4) Optional extras — leave as '' to skip
REF_POINTS = ''   # '' = camera-relative mode (any footage); or 'refs.json' from cell 4a
ROSTER = ''       # roster.csv: jersey_number,name

In [ ]:
# 5) Run the pipeline on GPU (yolov8x is auto-selected when CUDA is available)
import os, shlex
os.environ['SCOUT_DEVICE'] = 'cuda'
args = list(SOURCE)
if REF_POINTS: args += ['--ref-points', REF_POINTS]
if ROSTER: args += ['--roster', ROSTER]
cmd = shlex.join(args)   # quotes any path containing spaces
!python -m scout.pipeline {cmd}

In [ ]:
# 6) Package + download results (extract into your LOCAL project's data/ folder)
import pathlib, os
data = pathlib.Path('data')
matches = sorted(data.glob('matches/*/metrics.parquet'))
print('analyzed matches found:', len(matches))
if not matches:
    print('Nothing to package — cell 5 has not produced results yet. Check its output for errors.')
else:
    !rm -f /content/scout_results.zip
    !cd data && zip -qr /content/scout_results.zip . -x 'videos/*' 'uploads/*'
    z = pathlib.Path('/content/scout_results.zip')
    print(f'{z} — {z.stat().st_size/1e6:.1f} MB')
    !unzip -l /content/scout_results.zip | tail -15
    from google.colab import files
    files.download(str(z))   # if the browser blocks it, use the folder icon in the left sidebar

## 6b) See the results right here (no download needed)

In [ ]:
# 6b) Print ratings straight from the Colab database
import pandas as pd, json, pathlib
from scout.config import get_settings
from scout.db import Match, Player, Rating, get_session

pd.set_option('display.width', 200)
with get_session() as db:
    matches = db.query(Match).all()
    if not matches:
        print('No matches in the database yet — run cell 5 first.')
    for m in matches:
        players = db.query(Player).filter_by(match_id=m.id).all()
        ratings = {r.player_id: r for r in db.query(Rating).filter_by(match_id=m.id).all()}
        mode = json.loads((get_settings().match_dir(m.id) / 'projection.json').read_text())['mode']
        print(f"\n=== {m.id} | {m.source[-60:]} | mode: {mode} ===")
        if mode == 'relative':
            print('camera-relative: on-ball KPIs only; distance/speed/position metrics excluded\n')
        qf = get_settings().match_dir(m.id) / 'quality.json'
        if qf.exists():
            q = json.loads(qf.read_text())
            print(f"tracking: {q['n_raw_tracks']} raw tracks -> {q['n_rated']} rated "
                  f"({q['n_jerseys_read']} jersey numbers read, "
                  f"{q['n_fragments_dropped']} fragments dropped)")
            if q['n_raw_tracks'] > 4 * max(q['n_rated'], 1):
                print('WARNING: heavy identity fragmentation — ratings unreliable. '
                      'Use continuous, fixed-camera footage rather than edited highlights.')
            print()
        rows = [{'Player': p.name, 'Jersey': p.jersey, 'Team': p.team, 'Pos': p.role,
                 'PosConf': round(p.role_confidence or 0, 2), 'Mins': round(p.minutes, 1),
                 'Rating': ratings[p.id].overall if p.id in ratings else None,
                 **(ratings[p.id].sub_scores or {} if p.id in ratings else {})}
                for p in players]
        df = pd.DataFrame(rows).sort_values('Rating', ascending=False, na_position='last')
        display(df.reset_index(drop=True))

        print('Best by position:')
        for role, label in {'GK': 'Goalkeeper', 'DEF': 'Defender',
                            'MID': 'Midfielder', 'ATT': 'Attacker'}.items():
            c = [(p, ratings[p.id]) for p in players
                 if p.role == role and p.id in ratings and ratings[p.id].overall is not None]
            best = max(c, key=lambda x: x[1].overall) if c else None
            print(f'  {label:11s} ' + (f'{best[0].name} (#{best[0].jersey}) -> {round(best[1].overall)}'
                                       if best else '—'))
        top = max((r for r in ratings.values() if r.note), key=lambda r: r.overall, default=None)
        if top:
            print('\nScouting note (top player):\n', top.note)

## 7) OPTIONAL — open the dashboard from inside Colab

Colab has no browser access to `localhost`, so Streamlit is exposed through a free cloudflared tunnel (no account needed). The cell keeps running while the dashboard is up — **click the `https://….trycloudflare.com` link it prints**, and press the stop button when finished.

Coach corrections you make here are written to the Colab copy of the database, so re-download the results zip (cell 6) afterwards if you want to keep them.

In [ ]:
# 7) Serve the Streamlit dashboard through a public tunnel
!pip -q install -e ".[ui]" 2>&1 | tail -1
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
      -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# start Streamlit in the background (log: /content/streamlit.log)
get_ipython().system_raw(
    'streamlit run app/dashboard.py --server.port 8501 --server.headless true '
    '--server.fileWatcherType none &> /content/streamlit.log &')

import time; time.sleep(8)
print('Click the trycloudflare.com link below (stop this cell to shut the dashboard down)\n')
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate 2>&1 | grep -v '^$'

## 4a) Pitch reference points — OPTIONAL, run only for fixed wide-angle footage

Skip unless the camera is fixed and the whole pitch is visible. Without reference points the pipeline still rates players on on-ball actions (touches, passing, duels, possession won/lost, shots); distance, speed and pitch-position metrics are excluded rather than guessed.

To use: run cell 5 once (it ingests the video), then run the two cells below and re-run cell 5 with `REF_POINTS = 'refs.json'`.

Pitch coordinates: origin `[0, 0]` = left-bottom corner as seen from the camera, pitch 100 m x 64 m — corners are `[0,0]`, `[100,0]`, `[100,64]`, `[0,64]`.

In [ ]:
# 4a-i) Show an ingested frame with pixel axes, to read landmark coordinates off
import cv2, glob, matplotlib.pyplot as plt
vids = sorted(glob.glob('data/matches/*/video.mp4'))
if not vids:
    print('No ingested video yet — run cell 5 first, or skip this section entirely.')
else:
    cap = cv2.VideoCapture(vids[-1])
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT) * 0.5))
    ok, frame = cap.read(); cap.release()
    plt.figure(figsize=(16, 9))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.grid(color='yellow', alpha=0.4)
    plt.xticks(range(0, frame.shape[1], 50), rotation=90, fontsize=6)
    plt.yticks(range(0, frame.shape[0], 50), fontsize=6)
    plt.show()
    print('frame size (w,h):', frame.shape[1], frame.shape[0])

In [ ]:
# 4a-ii) Fill in the pixel coords you read above, then run to write refs.json
import json
refs = {'points': [
    {'px': [112, 596],  'pitch': [0, 0]},      # <-- left-bottom corner
    {'px': [1163, 601], 'pitch': [100, 0]},    # <-- right-bottom corner
    {'px': [986, 82],   'pitch': [100, 64]},   # <-- right-top corner
    {'px': [297, 80],   'pitch': [0, 64]},     # <-- left-top corner
]}
open('refs.json', 'w').write(json.dumps(refs))
REF_POINTS = 'refs.json'
print('wrote refs.json — now re-run cell 5')